In [ ]:
import pandas as pd
import numpy as np
import json
import os

In [ ]:
import pandas as pd
import numpy as np
import json

# =========================================
# Fixed pipeline: prune -> impute -> bin
# =========================================

# === 1. Load dataset ===
df = pd.read_csv("../data/mcs_sh_sample_Feb_28.csv")
df.columns = df.columns.str.strip()

# Normalize common missing markers
df = df.replace(["", " ", "NA", "N/A", "null", "nan"], np.nan)

#Fix cannabis variable
if "child_cannabis" in df.columns:
    df["child_cannabis"] = df["child_cannabis"].fillna("Never")
    print("✅ Re-coded child_cannabis missing values to 'Never' (structural missingness).")

# Fix alcohol variable
if "child_alcohol" in df.columns:
    df["child_alcohol"] = df["child_alcohol"].fillna("none")
    print("✅ Re-coded child_alcohol missing values to 'none' (structural missingness).")

# Coerce numeric-like columns to numeric when possible (does not touch true strings)
df.iloc[:, 2:] = df.iloc[:, 2:].apply(pd.to_numeric, errors="ignore")

# === 2. Define target and raw features ===
target_col = "suicide_17y"
target = df[target_col].copy()

X_raw = df.iloc[:, 2:].copy()

# === 3. Separate numeric and categorical based on dtypes after coercion ===
numeric_cols = X_raw.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = X_raw.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

print(f"Numeric features: {len(numeric_cols)}, Categorical features: {len(categorical_cols)}")

# ---------------------------------------------------------
# Step 1. Correlation based pruning on RAW numeric features
# Pairwise complete correlations (no imputation yet)
# ---------------------------------------------------------

def remove_highly_correlated_features_pairwise(df_num: pd.DataFrame, threshold: float = 0.9, min_pairwise_n: int = 50):
    cols = df_num.columns.tolist()
    to_drop = set()

    for i, col_i in enumerate(cols):
        if col_i in to_drop:
            continue
        s_i = df_num[col_i]
        for j in range(i + 1, len(cols)):
            col_j = cols[j]
            if col_j in to_drop:
                continue

            pair = pd.concat([s_i, df_num[col_j]], axis=1).dropna()
            if pair.shape[0] < min_pairwise_n:
                continue

            r = pair.iloc[:, 0].corr(pair.iloc[:, 1])
            if pd.notna(r) and abs(r) > threshold:
                to_drop.add(col_j)

    reduced = df_num.drop(columns=list(to_drop), errors="ignore")
    return reduced, sorted(list(to_drop))

numeric_raw = X_raw[numeric_cols].copy()
numeric_pruned_raw, dropped_correlated_numeric_fixed = remove_highly_correlated_features_pairwise(
    numeric_raw, threshold=0.9, min_pairwise_n=50
)

kept_numeric_cols = numeric_pruned_raw.columns.tolist()
print(f"🧹 Dropped {len(dropped_correlated_numeric_fixed)} raw numeric features due to high pairwise correlation.")

# ---------------------------------------------------------
# IMPORTANT FIX: rebuild working feature table using ONLY kept cols
# ---------------------------------------------------------
kept_feature_cols = kept_numeric_cols + categorical_cols
X_work = X_raw[kept_feature_cols].copy()

# ---------------------------------------------------------
# Step 2. Imputation (after pruning, on X_work only)
# ---------------------------------------------------------

# Numeric median imputation (vectorized and robust)
X_work[kept_numeric_cols] = X_work[kept_numeric_cols].apply(lambda s: s.fillna(s.median()))

# Categorical mode imputation
for col in categorical_cols:
    mode_val = X_work[col].mode(dropna=True)
    fill_val = mode_val.iloc[0] if not mode_val.empty else "Unknown"
    X_work[col] = X_work[col].fillna(fill_val)

print(f"✅ Applied median imputation to {len(kept_numeric_cols)} numeric columns (after pruning).")
print(f"✅ Applied mode imputation to {len(categorical_cols)} categorical columns (after pruning).")

# Sanity check
remaining = X_work.isna().sum()
remaining = remaining[remaining > 0].sort_values(ascending=False)
print("Remaining NaNs after imputation (should be empty):")
print(remaining)
# ---------------------------------------------------------
# SAVE UNBINNED DATASET FOR ODDS RATIOS (LOGISTIC REGRESSION)
# ---------------------------------------------------------

df_unbinned_or = X_work.copy()
df_unbinned_or[target_col] = target.values

df_unbinned_or.to_csv(
    "../data/mcs_sh_sample_Feb_28_OR.csv",
    index=False
)

print("📁 Unbinned dataset for odds ratios saved.")
# ---------------------------------------------------------
# Step 3. Binning (last, after imputation)
# ---------------------------------------------------------

df_binned_fixed = X_work.copy()
bin_edges_fixed = {}

# Candidates based on kept numeric cols
candidates = [c for c in kept_numeric_cols if df_binned_fixed[c].nunique() > 5]
print(f"Numeric candidates for binning: {len(candidates)}")

for col in candidates:
    try:
        binned, bins = pd.qcut(df_binned_fixed[col], q=4, duplicates="drop", retbins=True)
        n_bins = binned.cat.categories.size

        if n_bins != 4:
            print(f"⚠️ Skipping {col}: only {n_bins} bins could be formed")
            continue

        new_col = f"{col}_binned"
        df_binned_fixed[new_col] = (binned.cat.codes + 1).astype(float)  # 1..4
        df_binned_fixed.drop(columns=[col], inplace=True)
        bin_edges_fixed[col] = bins.tolist()
        print(f"✅ Binned {col} into {new_col}")

    except ValueError as e:
        print(f"⚠️ Skipping {col}: {e}")

# Save binned variables and metadata (fixed)
binned_only_fixed = df_binned_fixed.filter(regex="_binned$")
binned_only_fixed.to_excel(
    "../preprocessing_files/binned_variables_sh.xlsx",
    index=False
)
with open("../preprocessing_files/binning_metadata_sh.json", "w") as f:
    json.dump(bin_edges_fixed, f, indent=2)

print("📁 Binning info saved (fixed).")
# ---------------------------------------------------------
# Enforce meaningful reference groups BEFORE one hot encoding
# ---------------------------------------------------------

import pandas as pd

# child alcohol → reference = none
if "child_alcohol" in df_binned_fixed.columns:
    df_binned_fixed["child_alcohol"] = pd.Categorical(
        df_binned_fixed["child_alcohol"],
        categories=["none", "some", "many"],
        ordered=False
    )

# child cannabis → reference = Never
if "child_cannabis" in df_binned_fixed.columns:
    df_binned_fixed["child_cannabis"] = pd.Categorical(
        df_binned_fixed["child_cannabis"],
        categories=["Never", "one to four", "more than 5"],
        ordered=False
    )

# maternal alcohol → reference = low risk
if "mAlcohol_binary" in df_binned_fixed.columns:
    df_binned_fixed["mAlcohol_binary"] = pd.Categorical(
        df_binned_fixed["mAlcohol_binary"],
        categories=["low risk", "high risk"],
        ordered=False
    )

# paternal alcohol → reference = low risk
if "fAlcohol_binary" in df_binned_fixed.columns:
    df_binned_fixed["fAlcohol_binary"] = pd.Categorical(
        df_binned_fixed["fAlcohol_binary"],
        categories=["low risk", "high risk"],
        ordered=False
    )

# maternal education → reference = lower edu
if "mEdu" in df_binned_fixed.columns:
    df_binned_fixed["mEdu"] = pd.Categorical(
        df_binned_fixed["mEdu"],
        categories=["lower edu", "higher edu", "overseas"],
        ordered=False
    )
# ---------------------------------------------------------
# One hot encode categorical columns (after binning)
# ---------------------------------------------------------

categorical_cols_for_encoding = [c for c in categorical_cols if c in df_binned_fixed.columns]
df_encoded_fixed = pd.get_dummies(df_binned_fixed, columns=categorical_cols_for_encoding, drop_first=True)
print(f"✅ One hot encoded {len(categorical_cols_for_encoding)} categorical columns (fixed).")

# Add target back
df_final_fixed = df_encoded_fixed.copy()
df_final_fixed[target_col] = target.values

# Save dropped list
with open("../preprocessing_files/dropped_correlated_features_sh.json", "w") as f:
    json.dump(dropped_correlated_numeric_fixed, f, indent=2)

print(f"🧹 Dropped {len(dropped_correlated_numeric_fixed)} correlated numeric features (fixed).")

# Final sanity check on output
final_remaining = df_final_fixed.drop(columns=[target_col]).isna().sum()
final_remaining = final_remaining[final_remaining > 0].sort_values(ascending=False)
print("Remaining NaNs in final dataset (should be empty):")
print(final_remaining)

# Save final dataset
df_final_fixed.to_csv(
    "../data/mcs_sh_sample_Feb_28_preprocessed.csv",
    index=False
)
print("✅ Final preprocessed dataset saved (fixed).")